In [15]:
"""
Car Price Prediction - Data Generation and Model Training
This script generates a synthetic dataset of car prices, processes the data,
trains machine learning models, and saves the final prediction pipeline.
"""

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [16]:
# ==========================================
# 1. DATA GENERATION
# ==========================================
# Generate a synthetic dataset for 1000 cars
np.random.seed(42)
num_samples = 1000

data = {
    'brand': np.random.choice(['Toyota', 'BMW', 'Mercedes', 'Hyundai', 'Kia'], num_samples),
    'year': np.random.randint(2005, 2024, num_samples),
    'engine': np.random.choice([1.6, 2.0, 2.5, 3.0, 4.4], num_samples),
    'mileage': np.random.randint(0, 300000, num_samples),
    'fuel': np.random.choice(['Petrol', 'Diesel'], num_samples)
}

df = pd.DataFrame(data)

# Logical pricing formula (This defines the underlying pattern the model should learn)
# Brand factor: Premium brands (Mercedes/BMW) are expensive, economy brands (Hyundai/Kia) are cheaper
brand_factor = df['brand'].map({'Toyota': 1.0, 'BMW': 1.5, 'Mercedes': 1.6, 'Hyundai': 0.8, 'Kia': 0.75})

# Year factor: Newer cars are more expensive
year_factor = (df['year'] - 2000) * 1000

# Engine factor: Larger engine volumes increase the price
engine_factor = df['engine'] * 5000

# Mileage factor: Higher mileage decreases the price
mileage_factor = df['mileage'] * -0.05

# Fuel factor: Diesel cars have a slight price premium
fuel_factor = df['fuel'].map({'Petrol': 0, 'Diesel': 2000})

# Calculate final price and add random noise to simulate real-world variance
df['price'] = (10000 * brand_factor) + year_factor + engine_factor + mileage_factor + fuel_factor + np.random.normal(0, 2000, num_samples)
df['price'] = df['price'].astype(int)  # Convert prices to integers

# Remove anomalies (e.g., negative/extremely low prices)
df = df[df['price'] > 2000]

# Save the generated dataset to a CSV file
df.to_csv('cars_data.csv', index=False)
print("Data generation complete. Saved as: cars_data.csv")

Data generation complete. Saved as: cars_data.csv


In [17]:
# ==========================================
# 2. DATA PREPROCESSING
# ==========================================
numeric_columns = ['year', 'engine', 'mileage']
categorical_columns = ['brand', 'fuel']

# Define the column transformer to scale numeric features and encode categorical ones
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first'), categorical_columns)
    ]
)

# Separate features (X) and target variable (y)
X = df.drop(columns=['price'])
y = df['price']

# Apply transformations
X_scaled = preprocessor.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [18]:
# ==========================================
# 3. MODEL TRAINING & EVALUATION
# ==========================================
# Initialize the models
model_lr = LinearRegression()
model_rfr = RandomForestRegressor()

# Train the models
model_lr.fit(X_train, y_train)
model_rfr.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_lr = model_lr.predict(X_test)
y_pred_rfr = model_rfr.predict(X_test)

# Evaluate model performance using Mean Absolute Error (MAE)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mae_rfr = mean_absolute_error(y_test, y_pred_rfr)

print(f'Linear Regression MAE : {mae_lr:.2f}')
print(f'Random Forest MAE     : {mae_rfr:.2f}')

Linear Regression MAE : 1475.25
Random Forest MAE     : 2529.87


In [19]:
# ==========================================
# 4. DEPLOYMENT PIPELINE
# ==========================================
# Create a unified pipeline consisting of the preprocessor and the linear regression model
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ]
)

# Fit the pipeline on the entire dataset to maximize training information before saving
pipeline.fit(X, y)

# Serialize and save the pipeline for later use in applications
joblib.dump(pipeline, 'car_price_prediction.joblib')

['car_price_prediction.joblib']